# 🧠 CalRetail — Competitor Price Monitoring
## Goal
Identifies structural pricing discrepancies, flags outlier alerts using Z-Score calculations,
and suggests a concrete target price for each flagged SKU.

## Algorithmic Explanation
**Statistical Outlier Alert Pipeline**
1. Cross-reference internal item records with corresponding competitor pricing lists.
2. Compute pricing percentage variance (`price_gap = (our_price - comp_avg) / comp_avg`).
3. Calculate distribution parameters (mean, standard deviation) for pricing gaps.
4. Apply Z-score classification triggers (±1.5σ, consistent with
   `adaptive_thresholds.get_competitor_gap_stats`) and propose a concrete target price for
   flagged items.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
cust_pr = load_table('competitor_pricing')
prod = load_table('products')

# Calculate mean competitor pricing per SKU
comp_means = cust_pr.groupby('product_id')['price'].mean().reset_index()
comp_means.rename(columns={'price': 'comp_avg_price'}, inplace=True)

# Merge
pricing_merge = pd.merge(prod, comp_means, on='product_id')
pricing_merge['price_gap_pct'] = ((pricing_merge['price'] - pricing_merge['comp_avg_price']) / pricing_merge['comp_avg_price']) * 100

mean_gap = pricing_merge['price_gap_pct'].mean()
std_gap = pricing_merge['price_gap_pct'].std()
print(f"Price gap distribution. Mean: {mean_gap:.2f}% | Std Dev: {std_gap:.2f}%")


In [ ]:
def detect_pricing_outliers():
    results = []
    for idx, row in pricing_merge.iterrows():
        gap = row['price_gap_pct']
        z_score = (gap - mean_gap) / (std_gap if std_gap > 0 else 1)
        
        if z_score > 1.5:
            status = "Overpriced"
            action = "Reduce price to align with competition"
            suggested_price = round(float(row['comp_avg_price']) * 1.02, 2)  # small premium over comp avg
        elif z_score < -1.5:
            status = "Underpriced"
            action = "Opportunity to raise price and gain margin"
            suggested_price = round(float(row['comp_avg_price']) * 0.98, 2)  # stay competitive, capture margin
        else:
            status = "Optimal"
            action = "Maintain current pricing"
            suggested_price = round(float(row['price']), 2)
            
        results.append({
            "product_id": row['product_id'],
            "product_name": row['product_name'],
            "our_price": float(row['price']),
            "competitor_mean": float(row['comp_avg_price']),
            "gap_pct": round(float(gap), 2),
            "z_score": round(float(z_score), 2),
            "status": status,
            "recommended_action": action,
            "suggested_price": suggested_price,
        })
    return results

alerts = detect_pricing_outliers()
overpriced = [s for s in alerts if s['status'] == "Overpriced"]
print(f"Detected {len(overpriced)} overpriced items. Sample alert:\n", json.dumps(overpriced[0] if overpriced else alerts[0], indent=2))

In [ ]:
print("=== CALRETAIL COMPETITOR MONITORING BOARD ===")
alerts_df = pd.DataFrame(alerts)
print("Summary Statistics:")
print(alerts_df['status'].value_counts())
print("\nTop 5 Items Requiring Margin Action:")
print(alerts_df[alerts_df['status'] != 'Optimal'][['product_name', 'gap_pct', 'status', 'recommended_action']].head(5).to_string(index=False))
